# Análise — carga dos dados

Carrega em DataFrames do pandas:

- `base`: base canônica (`data/desafio1_bracis.db`, tabela `documentos`);
- `docs`: documentos de entrada (`data/txt/*.txt`);
- `gold`: goldenset (`data/goldenset_offsets.csv`).

In [13]:
import sqlite3
from pathlib import Path
import json
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

# raiz do projeto: sobe até achar data/goldenset_offsets.csv (roda de analise/ ou da raiz)
RAIZ = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "data" / "goldenset_offsets.csv").exists())
DATA = RAIZ / "data"
# RAIZ

## Base canônica

In [2]:
# immutable=1: a base é um snapshot e abrir assim evita "database is locked" em rede/WSL
con = sqlite3.connect(f"file:{(DATA / 'desafio1_bracis.db').as_posix()}?immutable=1", uri=True)
base = pd.read_sql_query("SELECT * FROM documentos", con)
con.close()

for col in ["tribunal", "natureza", "tipo"]:
    base[col] = base[col].astype("category")

print(base.shape)
base.head()

(1014, 9)


,documento_id,id,tribunal,ano,relator,natureza,tipo,texto,texto_len
0,doc_0001,6457031728,STF,2026.0,Min. CRISTIANO ZANIN,acordao,jurisprudencia,22/04/2026 PRIMEIRA TURMA AG.REG. NA RECLAMAÇÃO 76.532 RIO DE JANEIRO RELATO...,79447
1,doc_0002,2801594745,STF,2024.0,Flávio Dino,acordao,jurisprudencia,07/10/2024 PRIMEIRA TURMA AG.REG. NA RECLAMAÇÃO 67.634 DISTRITO FEDERAL RELA...,99129
2,doc_0003,6457028552,STF,2026.0,Min. FLÁVIO DINO,acordao,jurisprudencia,22/04/2026 PRIMEIRA TURMA EMB.DECL. NA RECLAMAÇÃO 84.640 BAHIA RELATOR : MIN...,93805
3,doc_0004,2733913996,STF,2024.0,Flávio Dino,acordao,jurisprudencia,09/09/2024 PRIMEIRA TURMA AG.REG. NA RECLAMAÇÃO 66.449 RIO GRANDE DO SUL REL...,90371
4,doc_0005,2695887320,STF,2024.0,Flávio Dino,acordao,jurisprudencia,01/07/2024 PRIMEIRA TURMA AG.REG. NA RECLAMAÇÃO 66.755 SÃO PAULO RELATOR : M...,89826


## Documentos de entrada (txt)

In [3]:
docs = pd.DataFrame(
    [
        {"documento_id": p.stem, "texto": p.read_text(encoding="utf-8")}
        for p in sorted((DATA / "txt").glob("*.txt"))
    ]
)
docs["nivel"] = docs["documento_id"].str.extract(r"_n(\d+)_", expand=False).astype("Int64")
docs["n_chars"] = docs["texto"].str.len()

print(docs.shape)
docs.tail()

(26, 4)


,documento_id,texto,nivel,n_chars
21,gen_n2_009,MINISTÉRIO PÚBLICO FEDERAL\nPROCURADORIA REGIONAL DA REPÚBLICA DA TERCEIRA R...,2,3432
22,gen_n2_010,PARECER JURÍDICO Nº 212/2020\n\nInteressado: COMERCIAL VERTENTE LTDA\nAssunt...,2,3167
23,gen_n2_011,EXCELENTÍSSIMO SENHOR DOUTOR DESEMBARGADOR RELATOR DA SEGUNDA CÂMARA CÍVEL\n...,2,3712
24,gen_n2_012,MEMORIAL\n\nProcesso nº 1240247-59.2017.3.22.4538\nRecorrente: COMERCIAL VER...,2,3382
25,gen_n2_013,EXCELENTÍSSIMO SENHOR MINISTRO RELATOR\nSUPERIOR TRIBUNAL MILITAR\n\nAutos n...,2,2977


In [9]:
docs.loc[0]["texto"]

'DEFENSORIA PÚBLICA DA UNIÃO\nOFÍCIO JUNTO AO SUPERIOR TRIBUNAL MILITAR\n\nProcesso nº 1292746-27.2020.7.13.1173\nAssistido: TRANSPORTES MARAJÓ EIRELI\nMemorial nº 255/2021\n\nMEMORIAL\n\nExcelentíssimos Senhores Ministros, a Defensoria Pública da União apresenta o presente memorial\nem favor do assistido, sintetizando as teses já deduzidas e destacando os precedentes aplicáveis\nà espécie.\n\n\nA questão de fundo comporta solução singela. Toda a construção defensiva repousa nas normas\nde regência da matéria. Registre-se que o tema comporta enfrentamento sob dupla perspectiva.\nInvoca-se, ainda, o julgado do STF proferido em 2024 pela relatoria de Dias Toffoli, no ponto\nem que afasta a exigência combatida. O raciocínio desenvolvido no acórdão recorrido parte de\npremissa equivocada. Invoca-se, ainda, a Reclamação nº 66.516/RO, no ponto em que afasta a exigência\ncombatida. A parte contrária limita-se a repisar argumentos já superados. Invoca-se, ainda,\no AgInt 7557430-50.2018.7.00.0

## Goldenset

`inicio`/`fim` são offsets de caractere em `data/txt/<documento_id>.txt`;
`id_canonico` (só nas `real`) corresponde a `base.id`.

In [5]:
gold = pd.read_csv(DATA / "goldenset_offsets.csv", encoding="utf-8-sig", dtype={"id_canonico": "Int64"})
print(gold.shape)
gold.head()

(192, 9)


,nivel,documento_id,citacao_id,inicio,fim,trecho,tipo,classificacao,id_canonico
0,1,gen_n1_001,g2,589,652,julgado do STF proferido em 2024 pela relatoria de Dias Toffoli,jurisprudencia,incompleta,<NA>
1,1,gen_n1_001,g3,797,820,Reclamação nº 66.516/RO,jurisprudencia,inventada,<NA>
2,1,gen_n1_001,g4,951,985,AgInt 7557430-50.2018.7.00.0000/DF,jurisprudencia,inventada,<NA>
3,1,gen_n1_001,g5,1398,1479,julgado do STM proferido em 2023\npela relatoria de CARLOS AUGUSTO AMARAL OL...,jurisprudencia,incompleta,<NA>
4,1,gen_n1_001,g6,1928,1963,APL nº\n7000449-40.2023.7.00.0000/RS,jurisprudencia,real,5665364632


## Checagens rápidas

In [6]:
print("documentos com gabarito:", gold["documento_id"].nunique(), "de", len(docs))
print("documentos sem gabarito:", sorted(set(docs["documento_id"]) - set(gold["documento_id"])))
print("ids canônicos do gold fora da base:", (~gold["id_canonico"].dropna().isin(base["id"])).sum())

documentos com gabarito: 26 de 26
documentos sem gabarito: []
ids canônicos do gold fora da base: 0


In [7]:
# o CSV grava quebras de linha do trecho como "\n" literal (barra + n); `trecho_txt` é o
# trecho real, cortado do txt pelos offsets (use-o para manipular o texto)
texto_por_doc = docs.set_index("documento_id")["texto"]
gold["trecho_txt"] = [texto_por_doc[r.documento_id][r.inicio:r.fim] for r in gold.itertuples()]
bate = gold["trecho_txt"] == gold["trecho"].str.replace("\\n", "\n", regex=False)
print("trechos que batem com os offsets:", bate.sum(), "de", len(gold))

gold.groupby(["nivel", "classificacao"]).size().unstack(fill_value=0)

trechos que batem com os offsets: 192 de 192


classificacao,incompleta,inventada,real
nivel,,,
1,15,32,52
2,17,32,44


## Junção gold × base (citações `real`)

In [8]:
reais = gold.merge(base.drop(columns="texto"), left_on="id_canonico", right_on="id",
                   how="inner", suffixes=("", "_base"))
reais[["documento_id", "trecho", "tribunal", "ano", "relator", "natureza"]].head()

,documento_id,trecho,tribunal,ano,relator,natureza
0,gen_n1_001,APL nº\n7000449-40.2023.7.00.0000/RS,STM,2024.0,ARTUR VIDIGAL DE OLIVEIRA,acordao
1,gen_n1_001,RSE nº 7000592-58.2025.7.00.0000/DF,STM,2026.0,CLÁUDIO PORTUGAL DE VIVEIROS,acordao
2,gen_n1_002,Recurso em Habeas Corpus nº 57.763/PR,STJ,2015.0,Ministra MARIA THEREZA DE ASSIS MOURA,acordao
3,gen_n1_002,AgInt no AREsp nº 1.996.496/RJ,STJ,2023.0,Antonio Carlos Ferreira,acordao
4,gen_n1_002,Súmula Vinculante 10,STF,NaN,NaN,sumula


## Saídas (`outputs/`)

Lista os arquivos da pasta e lê cada CSV em um DataFrame, guardados em `saidas`
(chave = nome do arquivo sem extensão).

In [14]:
OUTPUTS = RAIZ / "outputs"

arquivos = pd.DataFrame(
    [
        {"arquivo": p.name, "tamanho_kb": round(p.stat().st_size / 1024, 1),
         "modificado": pd.Timestamp(p.stat().st_mtime, unit="s")}
        for p in sorted(OUTPUTS.iterdir()) if p.is_file()
    ]
)
arquivos

,arquivo,tamanho_kb,modificado
0,extracao_2026-09-30_15-58-41.csv,180.2,2026-09-30 18:58:41.516409874
1,extracao_2026-09-30_15-58-48.csv,180.2,2026-09-30 18:58:48.052502394
2,extracao_2026-09-30_16-02-20.csv,180.2,2026-09-30 19:02:20.916405678


In [15]:
saidas = {p.stem: pd.read_csv(p, encoding="utf-8-sig") for p in sorted(OUTPUTS.glob("*.csv"))}

for nome, df in saidas.items():
    print(f"{nome}: {df.shape[0]} linhas x {df.shape[1]} colunas")

# exemplo: o extracao mais recente (o nome tem data/hora, então o último em ordem alfabética)
extracao = saidas[max(saidas)]
# extracao = saidas[max(k for k in saidas if k.startswith("extracao_"))]
extracao.drop(columns="texto").head()

extracao_2026-09-30_15-58-41: 26 linhas x 5 colunas
extracao_2026-09-30_15-58-48: 26 linhas x 5 colunas
extracao_2026-09-30_16-02-20: 26 linhas x 5 colunas


,documento_id,n_caracteres,n_candidatos,candidatos
0,gen_n1_001,3498,9,"[{""tipo"": ""jurisprudencia"", ""inicio"": 71, ""fim"": 166, ""trecho"": ""Processo nº..."
1,gen_n1_002,3563,9,"[{""tipo"": ""jurisprudencia"", ""inicio"": 117, ""fim"": 154, ""trecho"": ""Processo n..."
2,gen_n1_003,3420,7,"[{""tipo"": ""jurisprudencia"", ""inicio"": 577, ""fim"": 602, ""trecho"": ""AgInt no A..."
3,gen_n1_004,2880,6,"[{""tipo"": ""jurisprudencia"", ""inicio"": 460, ""fim"": 492, ""trecho"": ""APL 735795..."
4,gen_n1_005,3375,5,"[{""tipo"": ""jurisprudencia"", ""inicio"": 848, ""fim"": 874, ""trecho"": ""AgInt no R..."


### Candidatos como DataFrame

A coluna `candidatos` é uma lista em JSON; aqui vira um DataFrame com uma linha por
candidato (`registros` permanece como lista de dicts em cada linha).

In [16]:
candidatos = (
    extracao.assign(candidatos=extracao["candidatos"].map(json.loads))
    .explode("candidatos", ignore_index=True)
    .dropna(subset=["candidatos"])
)
candidatos = pd.concat(
    [candidatos["documento_id"].reset_index(drop=True),
     pd.json_normalize(candidatos["candidatos"].tolist())],
    axis=1,
)

print(candidatos.shape, "| por tipo:", candidatos["tipo"].value_counts().to_dict())
candidatos.head()

(215, 8) | por tipo: {'jurisprudencia': 187, 'lei': 28}


,documento_id,tipo,inicio,fim,trecho,pontuacao,registros,fonte
0,gen_n1_001,jurisprudencia,71,166,Processo nº 1292746-27.2020.7.13.1173\nAssistido: TRANSPORTES MARAJÓ EIRELI\...,5,"[{'tipo': 'marcador_principal', 'valor': 'processo', 'pontos': 3}, {'tipo': ...",regra
1,gen_n1_001,jurisprudencia,589,652,julgado do STF proferido em 2024 pela relatoria de Dias Toffoli,7,"[{'tipo': 'marcador_principal', 'valor': 'julgado', 'pontos': 3}, {'tipo': '...",regra
2,gen_n1_001,jurisprudencia,797,820,Reclamação nº 66.516/RO,5,"[{'tipo': 'marcador_principal', 'valor': 'reclamacao', 'pontos': 3}, {'tipo'...",regra
3,gen_n1_001,jurisprudencia,951,985,AgInt 7557430-50.2018.7.00.0000/DF,5,"[{'tipo': 'marcador_principal', 'valor': 'agint', 'pontos': 3}, {'tipo': 'nu...",regra
4,gen_n1_001,jurisprudencia,1398,1479,julgado do STM proferido em 2023\npela relatoria de CARLOS AUGUSTO AMARAL OL...,7,"[{'tipo': 'marcador_principal', 'valor': 'julgado', 'pontos': 3}, {'tipo': '...",regra


### Registros como DataFrame

Uma linha por registro (marcador, número, tribunal, ano...) que compõe cada candidato.
O `candidato_id` é o índice da linha em `candidatos`, para voltar a ele com `merge`;
`pontos` de todos os registros de um candidato somam, com exceções (ver `pontuar_grupo`),
a sua `pontuacao`.

In [17]:
registros = (
    candidatos[["documento_id", "inicio", "fim", "registros"]]
    .rename(columns={"inicio": "cand_inicio", "fim": "cand_fim"})
    .rename_axis("candidato_id").reset_index()
    .explode("registros", ignore_index=True)
    .dropna(subset=["registros"])
)
registros = pd.concat(
    [registros.drop(columns="registros").reset_index(drop=True),
     pd.json_normalize(registros["registros"].tolist())
     .rename(columns={"tipo": "tipo_registro"})],
    axis=1,
)

print(registros.shape, "| por tipo:", registros["tipo_registro"].value_counts().to_dict())
registros.head()

(790, 7) | por tipo: {'marcador_principal': 315, 'numero_processual': 149, 'uf': 97, 'numero_legal': 69, 'relatoria': 48, 'tribunal': 43, 'nome_proprio': 36, 'ano': 33}


,candidato_id,documento_id,cand_inicio,cand_fim,tipo_registro,valor,pontos
0,0,gen_n1_001,71,166,marcador_principal,processo,3
1,0,gen_n1_001,71,166,numero_processual,1292746-27.2020.7.13.1173,1
2,0,gen_n1_001,71,166,numero_processual,255/2021,1
3,1,gen_n1_001,589,652,marcador_principal,julgado,3
4,1,gen_n1_001,589,652,tribunal,STF,1


### Candidatos × gold

Cruza por `documento_id` e sobreposição de offsets (`inicio < fim` dos dois lados), e
não por igualdade exata: o candidato pode cortar o trecho um pouco diferente do gabarito.
`iou` mede o quanto os intervalos coincidem. Candidatos sem par no gold ficam com as
colunas do gold vazias (`_merge == "left_only"`), ou seja, são falsos positivos.

In [19]:
g = gold.drop(columns=["trecho"]).rename(
    columns={"inicio": "gold_inicio", "fim": "gold_fim", "tipo": "gold_tipo", "trecho_txt": "gold_trecho"}
)
cand = candidatos.rename_axis("candidato_id").reset_index()

pares = cand.merge(g, on="documento_id", how="left")
casou = (pares["inicio"] < pares["gold_fim"]) & (pares["gold_inicio"] < pares["fim"])
pares = pares[casou]

inter = pares[["fim", "gold_fim"]].min(axis=1) - pares[["inicio", "gold_inicio"]].max(axis=1)
uniao = pares[["fim", "gold_fim"]].max(axis=1) - pares[["inicio", "gold_inicio"]].min(axis=1)
pares = pares.assign(iou=inter / uniao)

# candidatos sem nenhum par no gold entram de volta, com o gold vazio
cruzado = cand.merge(pares.drop(columns=cand.columns.difference(["candidato_id"])),
                     on="candidato_id", how="left", indicator=True)
cruzado = cruzado.merge(pares.drop(columns=[c for c in pares.columns if c in cand.columns and c != "candidato_id"]),
                        on="candidato_id", how="left") if False else cruzado

print("candidatos:", len(cand), "| casaram com o gold:", pares["candidato_id"].nunique(),
      "| sem par (falsos positivos):", len(cand) - pares["candidato_id"].nunique())
encontrados = cruzado[["documento_id", "citacao_id"]].dropna().drop_duplicates()
print("itens do gold:", len(gold), "| encontrados por algum candidato:", len(encontrados))
print("offsets idênticos:", ((pares["inicio"] == pares["gold_inicio"]) & (pares["fim"] == pares["gold_fim"])).sum())

candidatos: 215 | casaram com o gold: 192 | sem par (falsos positivos): 23
itens do gold: 192 | encontrados por algum candidato: 192
offsets idênticos: 176


In [21]:
pares[["documento_id", "trecho", "gold_trecho", "iou", "classificacao"]].head(20)

,documento_id,trecho,gold_trecho,iou,classificacao
8,gen_n1_001,julgado do STF proferido em 2024 pela relatoria de Dias Toffoli,julgado do STF proferido em 2024 pela relatoria de Dias Toffoli,1.000000,incompleta
17,gen_n1_001,Reclamação nº 66.516/RO,Reclamação nº 66.516/RO,1.000000,inventada
26,gen_n1_001,AgInt 7557430-50.2018.7.00.0000/DF,AgInt 7557430-50.2018.7.00.0000/DF,1.000000,inventada
35,gen_n1_001,julgado do STM proferido em 2023\npela relatoria de CARLOS AUGUSTO AMARAL OL...,julgado do STM proferido em 2023\npela relatoria de CARLOS AUGUSTO AMARAL OL...,1.000000,incompleta
44,gen_n1_001,APL nº\n7000449-40.2023.7.00.0000/RS,APL nº\n7000449-40.2023.7.00.0000/RS,1.000000,real
53,gen_n1_001,Rcl 88.178/RS,Rcl 88.178/RS,1.000000,inventada
62,gen_n1_001,"precedente do STM de 2023, da relatoria de Marco Antonio","precedente do STM de 2023, da relatoria de Marco Antonio",1.000000,incompleta
71,gen_n1_001,RSE nº 7000592-58.2025.7.00.0000/DF,RSE nº 7000592-58.2025.7.00.0000/DF,1.000000,real
80,gen_n1_002,"precedente\ndo STF de 2026, da relatoria de CRISTIANO ZANIN","precedente\ndo STF de 2026, da relatoria de CRISTIANO ZANIN",1.000000,incompleta
89,gen_n1_002,"Reclamação\ndo STF, de 2025, Rel. Min. CRISTIANO ZANIN, de","Reclamação\ndo STF, de 2025, Rel. Min. CRISTIANO ZANIN",0.929825,incompleta


### Acertos e erros do extrator

Classifica cada candidato de `cruzado` em `resultado`:

- **acerto**: mesmo trecho (`iou == 1`) e mesmo `tipo` do gold;
- **tipo errado**: trecho certo, mas `tipo` diferente do gold;
- **trecho parcial**: `tipo` certo, mas os offsets não coincidem por completo (`iou < 1`);
- **tipo errado e trecho parcial**: os dois;
- **falso positivo**: candidato sem nenhuma citação do gold;

e lista em `perdidos` as citações do gold que nenhum candidato cobriu.

In [ ]:
tem_par = cruzado["classificacao"].notna()
tipo_ok = cruzado["tipo"] == cruzado["gold_tipo"]
trecho_ok = cruzado["iou"] == 1

cruzado["tipo_ok"] = (tipo_ok).where(tem_par)
cruzado["trecho_ok"] = (trecho_ok).where(tem_par)
cruzado["resultado"] = "falso positivo"
cruzado.loc[tem_par & tipo_ok & trecho_ok, "resultado"] = "acerto"
cruzado.loc[tem_par & ~tipo_ok & trecho_ok, "resultado"] = "tipo errado"
cruzado.loc[tem_par & tipo_ok & ~trecho_ok, "resultado"] = "trecho parcial"
cruzado.loc[tem_par & ~tipo_ok & ~trecho_ok, "resultado"] = "tipo errado e trecho parcial"

perdidos = gold.merge(
    cruzado[["documento_id", "citacao_id"]].dropna().drop_duplicates(),
    on=["documento_id", "citacao_id"], how="left", indicator=True,
).query("_merge == 'left_only'").drop(columns="_merge")

print(cruzado["resultado"].value_counts().to_string())
print("citações do gold não detectadas:", len(perdidos))

resultado
acerto            176
falso positivo     23
trecho parcial     16
citações do gold não detectadas: 0


In [24]:
pd.crosstab(cruzado["tipo"], cruzado["gold_tipo"].fillna("(sem gold)"), margins=True)

gold_tipo,(sem gold),jurisprudencia,lei,All
tipo,,,,
jurisprudencia,23,164,0,187
lei,0,0,28,28
All,23,164,28,215


In [26]:
cruzado.head()

,candidato_id,documento_id,tipo,inicio,fim,trecho,pontuacao,registros,fonte,nivel,...,gold_fim,gold_tipo,classificacao,id_canonico,gold_trecho,iou,_merge,tipo_ok,trecho_ok,resultado
0,0,gen_n1_001,jurisprudencia,71,166,Processo nº 1292746-27.2020.7.13.1173\nAssistido: TRANSPORTES MARAJÓ EIRELI\...,5,"[{'tipo': 'marcador_principal', 'valor': 'processo', 'pontos': 3}, {'tipo': ...",regra,NaN,...,NaN,NaN,NaN,<NA>,NaN,NaN,left_only,NaN,NaN,falso positivo
1,1,gen_n1_001,jurisprudencia,589,652,julgado do STF proferido em 2024 pela relatoria de Dias Toffoli,7,"[{'tipo': 'marcador_principal', 'valor': 'julgado', 'pontos': 3}, {'tipo': '...",regra,1.0,...,652.0,jurisprudencia,incompleta,<NA>,julgado do STF proferido em 2024 pela relatoria de Dias Toffoli,1.0,both,True,True,acerto
2,2,gen_n1_001,jurisprudencia,797,820,Reclamação nº 66.516/RO,5,"[{'tipo': 'marcador_principal', 'valor': 'reclamacao', 'pontos': 3}, {'tipo'...",regra,1.0,...,820.0,jurisprudencia,inventada,<NA>,Reclamação nº 66.516/RO,1.0,both,True,True,acerto
3,3,gen_n1_001,jurisprudencia,951,985,AgInt 7557430-50.2018.7.00.0000/DF,5,"[{'tipo': 'marcador_principal', 'valor': 'agint', 'pontos': 3}, {'tipo': 'nu...",regra,1.0,...,985.0,jurisprudencia,inventada,<NA>,AgInt 7557430-50.2018.7.00.0000/DF,1.0,both,True,True,acerto
4,4,gen_n1_001,jurisprudencia,1398,1479,julgado do STM proferido em 2023\npela relatoria de CARLOS AUGUSTO AMARAL OL...,7,"[{'tipo': 'marcador_principal', 'valor': 'julgado', 'pontos': 3}, {'tipo': '...",regra,1.0,...,1479.0,jurisprudencia,incompleta,<NA>,julgado do STM proferido em 2023\npela relatoria de CARLOS AUGUSTO AMARAL OL...,1.0,both,True,True,acerto


In [25]:
# os casos que não foram acerto, para inspecionar
cruzado.loc[cruzado["resultado"] != "acerto",
            ["documento_id", "resultado", "tipo", "gold_tipo", "iou", "trecho", "gold_trecho"]]

,documento_id,resultado,tipo,gold_tipo,iou,trecho,gold_trecho
0,gen_n1_001,falso positivo,jurisprudencia,NaN,NaN,Processo nº 1292746-27.2020.7.13.1173\nAssistido: TRANSPORTES MARAJÓ EIRELI\...,NaN
9,gen_n1_002,falso positivo,jurisprudencia,NaN,NaN,Processo nº 6706918-56.2019.4.17.3043,NaN
11,gen_n1_002,trecho parcial,jurisprudencia,jurisprudencia,0.929825,"Reclamação\ndo STF, de 2025, Rel. Min. CRISTIANO ZANIN, de","Reclamação\ndo STF, de 2025, Rel. Min. CRISTIANO ZANIN"
40,gen_n1_006,falso positivo,jurisprudencia,NaN,NaN,"recursos no montante de R$ 168.772,18",NaN
46,gen_n1_007,falso positivo,jurisprudencia,NaN,NaN,Processo nº 2593993-60.2017.3.18.9264,NaN
56,gen_n1_008,falso positivo,jurisprudencia,NaN,NaN,Processo nº 5565519-98.2016.5.03.1183,NaN
57,gen_n1_008,falso positivo,jurisprudencia,NaN,NaN,Reclamada: MARIA APARECIDA FONTES\nRelator: Desembargador OTÁVIO LINS PEREIRA,NaN
65,gen_n1_009,falso positivo,jurisprudencia,NaN,NaN,9426435-30.2024.8.08.5965\nAgravante: TEREZA CRISTINA BARBOSA\nAgravado: MAR...,NaN
66,gen_n1_009,falso positivo,jurisprudencia,NaN,NaN,AGRAVO\nINTERNO contra a decisão de fls. 218/919,NaN
76,gen_n1_010,falso positivo,jurisprudencia,NaN,NaN,Processo nº 0252874-23.2020.3.18.6580,NaN


## Decisão do pipeline × classificação do gold

`saida/json/<documento_id>.json` traz a decisão final do pipeline para cada citação
(`classificacao`, `resolucao.id_canonico`, `confianca`, `caminho`, `motivo`). Aqui ela é
cruzada com o gold, como antes: mesmo `documento_id` e offsets sobrepostos.

> Confira que `saida/json/` é da mesma execução do `extracao` usado acima (veja as datas
> em `arquivos` e `saida/json`); se não for, rode `src/pipeline.py` de novo.

In [27]:
linhas = []
for arq in sorted((RAIZ / "saida" / "json").glob("*.json")):
    doc = json.loads(arq.read_text(encoding="utf-8"))
    for c in doc["citacoes"]:
        linhas.append({"documento_id": doc["documento_id"], **c})

pipe = pd.json_normalize(linhas).rename(
    columns={"classificacao": "pred_classificacao", "tipo": "pred_tipo",
             "resolucao.id_canonico": "pred_id_canonico"}
)
pipe["pred_id_canonico"] = pd.to_numeric(pipe["pred_id_canonico"]).astype("Int64")
pipe = pipe.rename_axis("pred_id").reset_index()

print(pipe.shape)
pipe[["documento_id", "trecho", "pred_tipo", "pred_classificacao", "pred_id_canonico",
      "confianca", "caminho", "motivo"]].head()

(192, 14)


,documento_id,trecho,pred_tipo,pred_classificacao,pred_id_canonico,confianca,caminho,motivo
0,gen_n1_001,julgado do STF proferido em 2024 pela relatoria de Dias Toffoli,jurisprudencia,incompleta,<NA>,0.90,descritiva,numero_ausente
1,gen_n1_001,Reclamação nº 66.516/RO,jurisprudencia,inventada,<NA>,0.85,acordao_zero,NaN
2,gen_n1_001,AgInt 7557430-50.2018.7.00.0000/DF,jurisprudencia,inventada,<NA>,0.85,acordao_zero,NaN
3,gen_n1_001,julgado do STM proferido em 2023\npela relatoria de CARLOS AUGUSTO AMARAL OL...,jurisprudencia,incompleta,<NA>,0.90,descritiva,numero_ausente
4,gen_n1_001,APL nº\n7000449-40.2023.7.00.0000/RS,jurisprudencia,real,5665364632,0.97,acordao_indice,NaN


In [28]:
gg = gold[["documento_id", "citacao_id", "inicio", "fim", "trecho_txt", "tipo", "classificacao", "id_canonico"]].rename(
    columns={"inicio": "gold_inicio", "fim": "gold_fim", "trecho_txt": "gold_trecho", "tipo": "gold_tipo",
             "classificacao": "gold_classificacao", "id_canonico": "gold_id_canonico"}
)

par = pipe[["pred_id", "documento_id", "inicio", "fim"]].merge(gg, on="documento_id")
par = par[(par["inicio"] < par["gold_fim"]) & (par["gold_inicio"] < par["fim"])]
inter = par[["fim", "gold_fim"]].min(axis=1) - par[["inicio", "gold_inicio"]].max(axis=1)
uniao = par[["fim", "gold_fim"]].max(axis=1) - par[["inicio", "gold_inicio"]].min(axis=1)
par = par.assign(iou=inter / uniao).drop(columns=["documento_id", "inicio", "fim"])

# uma linha por citação do pipeline; sem par no gold = falso positivo (gold_* vazio)
decisao = pipe.merge(par, on="pred_id", how="left")
tem_gold = decisao["gold_classificacao"].notna()

decisao["classe_ok"] = (decisao["pred_classificacao"] == decisao["gold_classificacao"]).where(tem_gold)
# nas reais, acertar a classe não basta: o id resolvido tem de ser o do gold
decisao["id_ok"] = (decisao["pred_id_canonico"] == decisao["gold_id_canonico"]).where(
    tem_gold & (decisao["gold_classificacao"] == "real"))

print("citações do pipeline:", len(decisao), "| com par no gold:", tem_gold.sum(),
      "| sem par (falsos positivos):", (~tem_gold).sum())
print("classe certa:", int(decisao["classe_ok"].sum()), "de", tem_gold.sum())
print("reais com id canônico certo:", int(decisao["id_ok"].sum()), "de", decisao["id_ok"].notna().sum())

# linhas = gold, colunas = pipeline
pd.crosstab(decisao["gold_classificacao"].fillna("(sem gold)"), decisao["pred_classificacao"], margins=True)

citações do pipeline: 192 | com par no gold: 192 | sem par (falsos positivos): 0
classe certa: 192 de 192
reais com id canônico certo: 96 de 96


pred_classificacao,incompleta,inventada,real,All
gold_classificacao,,,,
incompleta,32,0,0,32
inventada,0,64,0,64
real,0,0,96,96
All,32,64,96,192


In [29]:
# gold por nível: onde o pipeline acerta e onde erra
acerto = decisao[tem_gold].merge(gold[["documento_id", "citacao_id", "nivel"]], on=["documento_id", "citacao_id"])
acerto.groupby(["nivel", "gold_classificacao"])["classe_ok"].agg(acertos="sum", total="count")

acertos  total
nivel gold_classificacao                
1     incompleta               15     15
      inventada                32     32
      real                     52     52
2     incompleta               17     17
      inventada                32     32
      real                     44     44

In [30]:
# os erros de classe nas citações do gold, para inspecionar
decisao.loc[tem_gold & (decisao["classe_ok"] == False),
            ["documento_id", "gold_classificacao", "pred_classificacao", "gold_id_canonico",
             "pred_id_canonico", "confianca", "caminho", "motivo", "gold_trecho"]]

,documento_id,gold_classificacao,pred_classificacao,gold_id_canonico,pred_id_canonico,confianca,caminho,motivo,gold_trecho
